# 🧪 MA-GRPO (A100 80GB): Sinh Thử 3 Outline + Chấm `MacroPlan-Bench` + Sinh 3 Dataset Train GRPO (Có Coordinator)
### Kiểm tra thực tế chất lượng dàn ý (chấm điểm tự động + Judge 32B) và 24 chapter prompts train GRPO trước khi chạy full 80 truyện

- **GPU mục tiêu:** Colab **A100 80GB**.
- **Input:** 3 đề bài MoPS (`data/mops_premises.jsonl`, 0% trùng tập test `tell-me-a-story-test.jsonl`).
- **Kiến trúc mới nhất (đã tối ưu sau Ablation):**
  - **Condition D (Macro Event Graph thuần nhân quả)**: Bỏ Freytag 5 pha, bỏ SubEventAgent.
  - **Đã xóa hoàn toàn Validator**: Không tốn inference LLM Validator hay fallback cứng.
  - **0-token Hygiene Gates**: Tự động chạy **Artifact Cleaner** + **Paragraph Deduplication Gate (>30 từ)** trên prose nối chương trước khi đưa vào Coordinator.
  - **Model Teacher (Outline + Transition Prose + Coordinator + Outline Judge):** `Qwen/Qwen2.5-32B-Instruct-AWQ` (`float16`, `max_model_len=16384`).
- **Đầu ra trên Google Drive (`/content/drive/MyDrive/MA-GRPO-Dataset-Sample3/`):**
  1. `grpo_sample_3mops_macro_plans.json` — 3 dàn ý đầy đủ (`Event Graph` + `Story Plan`).
  2. `outline_benchmark/` — Kết quả **Suite 1 (`MacroPlan-Bench`)** gồm `S_cov`, `S_causal`, `S_prog`, `S_comp`, **CONCOCT Pacing**, **32B DOC+HANNA Judge**, và **Perturbation Sanity Check**.
  3. `grpo_sample_3mops.jsonl` — **24 sample chapter prompts** (`3 truyện × 8 chương`, đã có `CoordinatorAgent` Two-Tier Memory + `prev_chapter_tail` 180 từ thật).
  4. `sample_3mops_readable_preview.md` — Bản Markdown trình bày đẹp cả 3 dàn ý + prompt mẫu để đọc trực tiếp.

---

## Bước 1: Kiểm tra GPU A100 80GB, Mount Drive & Sync Repo `MA-GRPO`

In [ ]:
!nvidia-smi

import os
import sys
import subprocess
import getpass
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_DIR = Path("/content/drive/MyDrive/MA-GRPO-Dataset-Sample3")
except Exception:
    OUTPUT_DIR = Path("sample_3mops_outputs")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Thư mục lưu kết quả: {OUTPUT_DIR}")

REPO_URL = "github.com/AIVIETNAM-AIO-TruongDogki/MA_GRPO.git"
REPO_DIR = "/content/MA-GRPO"

if os.path.exists("/content"):
    try:
        from google.colab import userdata
        token = userdata.get('GH_TOKEN')
    except Exception:
        token = None

    if not os.path.exists(REPO_DIR):
        if token:
            subprocess.run(["git", "clone", f"https://{token}@{REPO_URL}", REPO_DIR], check=True)
        else:
            res = subprocess.run(["git", "clone", f"https://{REPO_URL}", REPO_DIR])
            if res.returncode != 0:
                token = getpass.getpass("Nhập GitHub Personal Access Token: ")
                subprocess.run(["git", "clone", f"https://{token}@{REPO_URL}", REPO_DIR], check=True)
    else:
        subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)
    os.chdir(REPO_DIR)
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)

assert os.path.exists("scripts/generate_grpo_dataset.py"), "Không tìm thấy scripts/generate_grpo_dataset.py!"
assert os.path.exists("scripts/benchmark_outline.py"), "Không tìm thấy scripts/benchmark_outline.py!"
print("Working dir:", os.getcwd())
!git log -n 1 --oneline


## Bước 2: Cài đặt `vllm`, `sentence-transformers` & Thư viện Benchmark

In [ ]:
!pip install -q vllm transformers sentence-transformers accelerate matplotlib pandas pyyaml pydantic
print("✅ Cài đặt hoàn tất!")


## Bước 3: Xem Trước 3 Đề Bài MoPS Đầu Vào (`0%` trùng tập test)

In [ ]:
from src.dataset_generator import load_premises_pool

sample_premises, excluded = load_premises_pool(
    premises_files=["data/mops_premises.jsonl"],
    include_canonical=False,
    exclude_files=["data/tell-me-a-story-test.jsonl"],
    num_premises=3,
    target_words_choices=(8000, 9000, 10000),
    num_chapters=8,
)

print(f"Đã chọn {len(sample_premises)} đề bài MoPS (đã loại {excluded} trùng tập test):\n")
for i, p in enumerate(sample_premises, 1):
    print(f"[{i}] ID: {p['id']} | Genre: {p['genre']} | Target: {p['target_words']}w / {p['num_chapters']} chs")
    print(f"    Premise: {p['premise']}\n")


## Bước 4: Sinh 3 Outline Planning + 3 Dataset Train GRPO (24 Prompts có Coordinator Two-Tier Memory)
- Chạy trong subprocess tách biệt (`VLLM_WORKER_MULTIPROC_METHOD=spawn`) để vLLM giải phóng sạch 100% VRAM sau khi sinh xong.
- Tự động xuất song song:
  - `grpo_sample_3mops_macro_plans.json` (3 dàn ý đầy đủ cho Suite 1 `MacroPlan-Bench`)
  - `grpo_sample_3mops.jsonl` (24 chapter prompts train GRPO kèm `context_summary` + `long_term_memory` + `prev_chapter_tail` 180 từ thật).

In [ ]:
import subprocess
import sys

MODEL_ID = "Qwen/Qwen2.5-32B-Instruct-AWQ"
NUM_PREMISES = 3
NUM_CHAPTERS = 8
BATCH_SIZE = 3
OUTPUT_FILENAME = "grpo_sample_3mops.jsonl"
MACRO_PLANS_FILE = OUTPUT_DIR / "grpo_sample_3mops_macro_plans.json"

env = os.environ.copy()
env["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
env["VLLM_LOGGING_LEVEL"] = "WARNING"
env["PYTHONUNBUFFERED"] = "1"

cmd = [
    sys.executable, "-u", "scripts/generate_grpo_dataset.py",
    "--backend", "vllm",
    "--model", MODEL_ID,
    "--premises-file", "data/mops_premises.jsonl",
    "--no-canonical",
    "--exclude-test-file", "data/tell-me-a-story-test.jsonl",
    "--num-premises", str(NUM_PREMISES),
    "--num-chapters", str(NUM_CHAPTERS),
    "--target-words-choices", "8000", "9000", "10000",
    "--batch-size", str(BATCH_SIZE),
    "--gpu-memory-utilization", "0.88",
    "--max-model-len", "16384",
    "--prose-max-tokens", "1100",
    "--output-dir", str(OUTPUT_DIR),
    "--output-filename", OUTPUT_FILENAME,
]

print("Running dataset generation:\n", " ".join(cmd), "\n", flush=True)
proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
)
assert proc.stdout is not None
for line in proc.stdout:
    print(line, end="", flush=True)
ret = proc.wait()
if ret != 0:
    raise RuntimeError(f"generate_grpo_dataset.py exited with code {ret}")
assert MACRO_PLANS_FILE.exists(), f"Không thấy file dàn ý: {MACRO_PLANS_FILE}"
print(f"\n✅ Đã sinh xong 3 Macro Plans ({MACRO_PLANS_FILE.name}) và 24 GRPO chapter prompts!")


## Bước 5: Chấm Điểm Ngay 3 Outline Planning Bằng Suite 1 (`MacroPlan-Bench`)
Chạy đầy đủ cả 2 pha trên A100 80GB:
- **Phase 1**: `S_cov`, `S_causal` (DAG + DeBERTa-v3 NLI), `S_prog`, `S_comp`, **CONCOCT** (`mean_concreteness`, `pacing_balance_score`, `sparse_event_ratio`, `inter_event_diversity` MiniLM, `plan_critic_pass_rate`) + **Perturbation Sanity Check**.
- **Phase 2**: **DOC + HANNA Judge** 4 chiều (`outline_relevance`, `plot_coherence`, `surprise_and_twists`, `engagement_and_complexity`) bằng `Qwen/Qwen2.5-32B-Instruct-AWQ`.

In [ ]:
BENCH_OUT_DIR = OUTPUT_DIR / "outline_benchmark"

bench_cmd = [
    sys.executable, "-u", "scripts/benchmark_outline.py",
    "--inputs", str(MACRO_PLANS_FILE),
    "--labels", "condition_d_32b",
    "--output-dir", str(BENCH_OUT_DIR),
    "--judge-backend", "local_vllm",
    "--judge-model", MODEL_ID,
    "--judge-gpu-mem", "0.85",
    "--judge-max-len", "16384",
    "--run-sanity-check",
]

print("Running Suite 1 MacroPlan-Bench:\n", " ".join(bench_cmd), "\n", flush=True)
proc_b = subprocess.Popen(
    bench_cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
)
assert proc_b.stdout is not None
for line in proc_b.stdout:
    print(line, end="", flush=True)
ret_b = proc_b.wait()
if ret_b != 0:
    raise RuntimeError(f"benchmark_outline.py exited with code {ret_b}")

import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

display(Markdown((BENCH_OUT_DIR / "outline_benchmark_report.md").read_text(encoding="utf-8")))

raw_bench = [json.loads(l) for l in (BENCH_OUT_DIR / "outline_benchmark_raw.jsonl").read_text(encoding="utf-8").splitlines() if l.strip()]
df_b = pd.DataFrame(raw_bench)
cols_show = [
    "premise_id", "genre", "num_events", "planning_tokens",
    "S_cov", "S_causal", "S_prog", "S_comp",
    "mean_concreteness", "pacing_balance_score", "sparse_event_ratio",
    "inter_event_diversity", "judge_outline_overall", "judge_weakest_chapter",
]
print("\n=== CHI TIẾT ĐIỂM TỪNG OUTLINE (3 TRUYỆN) ===")
display(df_b[[c for c in cols_show if c in df_b.columns]])
for r in raw_bench:
    print(f"\n[{r['premise_id']} | {r['genre']}] Judge={r.get('judge_outline_overall')}/5 (weakest ch={r.get('judge_weakest_chapter')}): {r.get('judge_rationale')}")

# Biểu đồ 4 panel so sánh trực tiếp 3 truyện
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
panels = [
    ("S_causal", "Causal Integrity (S_causal)", (0, 1.05)),
    ("pacing_balance_score", "CONCOCT Pacing Balance", (0, 1.05)),
    ("mean_concreteness", "Mean Concreteness", (0, 1.05)),
    ("judge_outline_overall", "DOC+HANNA Judge (1-5)", (0, 5.2)),
]
labels_x = [f"{r['genre']}\n({r['premise_id'][:8]})" for r in raw_bench]
for ax, (col, title, ylim) in zip(axes, panels):
    vals = df_b[col].fillna(0)
    ax.bar(labels_x, vals, color=["#2b6cb0", "#2f855a", "#805ad5"][:len(vals)])
    ax.set_ylim(*ylim)
    ax.set_title(title)
plt.tight_layout()
plt.savefig(BENCH_OUT_DIR / "sample3_outline_chart.png", dpi=150)
plt.show()


## Bước 6: Soi Chi Tiết 3 Outline (`E1..E8`) & 24 Sample Dataset Train GRPO (Có Coordinator) + Xuất Markdown Đọc Thử

In [ ]:
from scripts.train import (
    extract_context_from_prompt,
    extract_tail_from_prompt,
    extract_target_words_from_text,
)

jsonl_path = OUTPUT_DIR / OUTPUT_FILENAME
report_path = OUTPUT_DIR / "dataset_quality_report.json"
macro_plans = json.loads(MACRO_PLANS_FILE.read_text(encoding="utf-8"))

print("=== DATASET QUALITY REPORT ===")
if report_path.exists():
    print(report_path.read_text(encoding="utf-8"))

rows = [json.loads(line) for line in jsonl_path.read_text(encoding="utf-8").splitlines() if line.strip()]
print(f"\nTổng số sample rows train GRPO: {len(rows)}")

table_rows = []
for r in rows:
    ctx_state = r.get("context_state") or {}
    ltm = ctx_state.get("long_term_memory") or {}
    prompt_txt = r["prompt"]
    table_rows.append({
        "sample_id": r["sample_id"][:16] + "..." if len(r["sample_id"]) > 18 else r["sample_id"],
        "genre": r.get("genre"),
        "ch": r.get("chapter_index"),
        "target_w": r.get("target_words"),
        "parser_ok": extract_target_words_from_text(prompt_txt) == r.get("target_words"),
        "ctx_w": len(str(r.get("context_summary", "")).split()),
        "tail_w": len(str(r.get("prev_chapter_tail", "")).split()),
        "tail_clean": (not r.get("prev_chapter_tail")) or (r["prev_chapter_tail"].strip()[-1] in '.!?"”’'),
        "ltm_events": len(ltm.get("major_events", [])),
        "ltm_in_prompt": "[LONG-TERM STORY MEMORY]:" in prompt_txt,
        "no_none_turn": "Land turn: None" not in prompt_txt,
        "no_freytag": "Narrative Arc Stage:" not in prompt_txt,
    })

df = pd.DataFrame(table_rows)
print("\n=== BẢNG TỔNG QUAN 24 SAMPLE CHAPTER PROMPTS (CÓ COORDINATOR) ===")
display(df)

# Xuất file Markdown đầy đủ cả 3 Outline + mẫu Dataset Train trên Drive
md_lines = ["# Sample Preview: 3 Outlines + 24 GRPO Chapter Prompts (Condition D, No Freytag, No Validator)\n"]
for idx, mp in enumerate(macro_plans, 1):
    it = mp.get("item", {})
    eg = mp.get("event_graph", {})
    md_lines.append(f"\n---\n## Outline {idx}: `{mp['premise_id']}` ({it.get('genre')})\n")
    md_lines.append(f"**Premise:** {it.get('premise')}\n")
    for ev in eg.get("events", []):
        deps = ", ".join(ev.get("depends_on", [])) or "None"
        md_lines.append(
            f"- **{ev.get('event_id')}: {ev.get('title')}** *(Depends On: `{deps}`)*\n"
            f"  - Setting: {ev.get('setting')}\n"
            f"  - Character: {ev.get('character')}\n"
            f"  - Action: {ev.get('action')}\n"
            f"  - Conflict: {ev.get('conflict')}\n"
            f"  - Plot Twist: {ev.get('plot_twist')}\n"
        )

seen_pids = []
for r in rows:
    pid = r["premise_id"]
    if pid not in seen_pids:
        seen_pids.append(pid)
        md_lines.append(f"\n---\n## GRPO Train Prompts — Story {len(seen_pids)}: `{pid}` ({r['genre']})\n")
    if r["chapter_index"] in (1, 2, 5, 8):
        md_lines.append(f"### Chapter {r['chapter_index']} — {r['chapter_title']} (target={r['target_words']}w)\n")
        md_lines.append(f"**Coordinator `context_summary`:**\n> {r['context_summary']}\n")
        if r["prev_chapter_tail"]:
            md_lines.append(f"**Coordinator `prev_chapter_tail` ({len(r['prev_chapter_tail'].split())} words):**\n> {r['prev_chapter_tail']}\n")
        md_lines.append(f"**Full GRPO Training Prompt:**\n```text\n{r['prompt']}\n```\n")

preview_md = OUTPUT_DIR / "sample_3mops_readable_preview.md"
preview_md.write_text("\n".join(md_lines), encoding="utf-8")
print(f"\n✅ Đã xuất bản đọc thử đầy đủ (3 Outlines + GRPO Prompts) tại: {preview_md}")

# In trực tiếp Outline 1 & Prompt Chương 2 của truyện 1 ngay trong cell
print("\n" + "=" * 75)
print("PREVIEW OUTLINE 1 (8 EVENTS CAUSAL DAG)")
print("=" * 75)
for ev in macro_plans[0]["event_graph"]["events"]:
    print(f"{ev['event_id']} [deps={ev.get('depends_on', [])}]: {ev['title']} -> Twist: {ev['plot_twist']}")

print("\n" + "=" * 75)
print("PREVIEW TRỰC TIẾP: TRUYỆN 1 — CHƯƠNG 2 (PROMPT TRAIN GRPO CÓ COORDINATOR + TAIL THẬT)")
print("=" * 75)
ch2 = next((r for r in rows if r["chapter_index"] == 2), rows[0])
print(ch2["prompt"])


## Bước 7: Tự động Ngắt Server Colab *(Mặc định `False` để bạn kịp đọc kết quả; đổi `True` nếu chạy không canh máy)*

In [ ]:
import time

AUTO_DISCONNECT = False

if AUTO_DISCONNECT:
    print("✅ Hoàn tất! Chờ 15 giây đồng bộ Drive rồi tự ngắt runtime...")
    time.sleep(15)
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception as e:
        print(f"Không thể gọi runtime.unassign(): {e}")
else:
    print("ℹ️ AUTO_DISCONNECT=False — Đọc xong nhớ tắt runtime thủ công để tiết kiệm Compute Units.")
